# FedQual-CPX: High-Performance GPU Benchmark Suite
### The Partial Observability Barrier in Dynamic Federated Client Selection

This notebook runs the complete multi-seed evaluations, Oort baseline, participation threshold sweep, and exploration weight ablations on a free **Google Colab T4 GPU**.

**Instructions**:
1. In the menu bar, go to **Runtime** → **Change runtime type**.
2. Set **Hardware accelerator** to **T4 GPU** and click **Save**.
3. Run the cells below sequentially.

In [ ]:
# 1. Verify GPU Availability
!nvidia-smi

In [ ]:
# 2. Clone Repository and Install Dependencies
!git clone https://github.com/Talhaasif7/FedQual-CPX.git
%cd FedQual-CPX
!git pull origin main
!pip install -r requirements.txt

In [ ]:
# 3. Download and Process CIFAR-10 Dataset
!python scripts/download_data.py --dataset cifar10

## Task 1: Main 7-Baseline Benchmark (CIFAR-10 Class Swap, 100 Rounds)
Runs 5 random seeds (42–46) across all 7 baselines:
- B0: Random / FedAvg
- B2: Utility Greedy
- B3: Sliding Window (W=10)
- B4: Fixed Exploration (eps=0.15)
- B6: Page-Hinckley Adaptive (Scaffold Control)
- B8: FedQual-CPX (Proposed CUSUM)
- B9: Oort (Lai et al., OSDI '21)

In [ ]:
!python experiments/run_main_experiments.py --drift-type class_swap

## Task 2: 10-Seed Statistical Validation Suite
Expands seeds from 5 to 10 (seeds 42 through 51) and computes:
- Paired Wilcoxon Signed-Rank Tests
- Paired Student's t-Tests
- Cohen's d Effect Sizes
- 95% Non-parametric Bootstrap Confidence Intervals

In [ ]:
!python experiments/run_multi_seed_evaluation.py --seeds 42 43 44 45 46 47 48 49 50 51

## Task 3: Participation Threshold Sweep (rho in {0.05, 0.10, 0.25, 0.36, 0.50})
Evaluates the renewal delay model threshold across the participation spectrum at N=100 for K in {5, 10, 25, 36, 50}.

In [ ]:
!python experiments/run_kn_sweep.py --k-list 5 10 25 36 50 --seeds 42 43 44 --num-rounds 100 --test-every 5

## Task 4: Exploration Weight & Guaranteed Priority Sweep
Tests detector inertness across w_c in {0.20, 0.50, 1.00, 2.00, 5.00} and the guaranteed exploration priority condition.

In [ ]:
!python experiments/run_weight_sweep.py --weights 0.20 0.50 1.00 2.00 5.00 --seeds 42 43 44 45 46 --num-rounds 100

## Task 5: Download Results Archive
Compacts all updated summary tables, CSVs, JSONs, and raw metrics into a single zip file and triggers a browser download.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('fedqual_cpx_colab_results', 'zip', 'results')
files.download('fedqual_cpx_colab_results.zip')